# Iron Rods

**Type:** Producer — stores output in dimensional depot  
**Blueprint:** `iron-rod` (60 Iron Ingots/min → 60 Iron Rods/min at 100%)  
**Downstream:** Rotors pulls from this storage

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS
import pandas as pd

BP  = BLUEPRINTS
TOL = 0.5  # items/min tolerance for balance assertions


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(f"  {r['machines']}× {r['building'].title()} → {r['per_machine_rate']:.2f} {name}/min each")
    return "<br>" + "<br>".join(lines)

## Constants

Set `IRON_INGOTS` to your allocation from the 270/min supply.  
Set `STASH_RATE` to how many Iron Rods/min should accumulate in storage — the Rotors notebook pulls the remainder.

In [ ]:
# ── Supply ────────────────────────────────────────────────────────────────────
IRON_INGOTS = None  # ← your ingot allocation (items/min)

# ── Stash target ──────────────────────────────────────────────────────────────
STASH_RATE = 20.0   # Iron Rods/min accumulating in storage

# ── Blueprint placement ───────────────────────────────────────────────────────
ROD_COPIES      = None  # ← number of blueprints to place
ROD_OUTPUT_EACH = None  # ← output rate per copy to set in-game (Iron Rods/min)

## Derived rates and balance

In [ ]:
assert None not in (IRON_INGOTS, ROD_COPIES, ROD_OUTPUT_EACH), \
    "Fill in all constants in the cell above before running this cell."

rod_total  = ROD_COPIES * ROD_OUTPUT_EACH
rod_ingots = rod_total  * BP['iron-rod'].input_ratio('iron-ingot', 'iron-rod')

available_for_rotors = rod_total - STASH_RATE

assert abs(rod_ingots - IRON_INGOTS) < TOL, (
    f"Ingot balance: consuming {rod_ingots:.2f}/min, supplied {IRON_INGOTS:.0f}/min "
    f"(delta {rod_ingots - IRON_INGOTS:+.2f})"
)
assert available_for_rotors > 0, (
    f"Not enough rods to stash {STASH_RATE}/min — only producing {rod_total:.2f}/min"
)

print(f"✓ Chain balanced")
print(f"  {IRON_INGOTS:.0f} Iron Ingots/min  →  {rod_total:.2f} Iron Rods/min")
print(f"  Stashing:              {STASH_RATE:.0f} Iron Rods/min")
print(f"  Available for Rotors:  {available_for_rotors:.2f} Iron Rods/min")